In [10]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import numpy as np

IMG_SIZE = (224, 224)
BATCH_SIZE = 32

# =========================
# LOAD DATA
# =========================
train_ds = keras.utils.image_dataset_from_directory(
    r"C:\Users\paula\OneDrive\Documentos\GitHub\Smart-Waste-Classification-NN-PROJECT-\data_processed\train",
    validation_split=0.2,
    subset="training",
    seed=42,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE
)

val_ds = keras.utils.image_dataset_from_directory(
    r"C:\Users\paula\OneDrive\Documentos\GitHub\Smart-Waste-Classification-NN-PROJECT-\data_processed\val",
    validation_split=0.2,
    subset="validation",
    seed=42,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE
)

class_names = train_ds.class_names
np.save("classes.npy", class_names)   # ✅ important fix
num_classes = len(class_names)

AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.cache().shuffle(1000).prefetch(AUTOTUNE)
val_ds = val_ds.cache().prefetch(AUTOTUNE)

# =========================
# MODEL
# =========================
base_model = keras.applications.MobileNetV2(
    input_shape=(*IMG_SIZE, 3),
    include_top=False,
    weights="imagenet"
)
base_model.trainable = False

model = keras.Sequential([
    layers.Input(shape=(*IMG_SIZE, 3)),

    # ⚠️ KEEP normalization ONLY HERE
    layers.Rescaling(1./127.5, offset=-1),

    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.1),
    layers.RandomZoom(0.1),

    base_model,
    layers.GlobalAveragePooling2D(),
    layers.Dropout(0.3),
    layers.Dense(num_classes, activation="softmax")
])

model.compile(
    optimizer=keras.optimizers.Adam(1e-3),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=15,
    callbacks=[
        keras.callbacks.EarlyStopping(patience=3, restore_best_weights=True)
    ]
)

model.save("waste_classifier_v2.keras")
print("✅ Model saved")

Found 5600 files belonging to 4 classes.
Using 4480 files for training.
Found 300 files belonging to 4 classes.
Using 60 files for validation.
Epoch 1/15
140/140 ━━━━━━━━━━━━━━━━━━━━ 43s 271ms/step - accuracy: 0.7683 - loss: 0.6398 - val_accuracy: 0.9500 - val_loss: 0.2788
Epoch 2/15
140/140 ━━━━━━━━━━━━━━━━━━━━ 37s 262ms/step - accuracy: 0.9288 - loss: 0.2228 - val_accuracy: 0.9833 - val_loss: 0.1287
Epoch 3/15
140/140 ━━━━━━━━━━━━━━━━━━━━ 34s 241ms/step - accuracy: 0.9498 - loss: 0.1635 - val_accuracy: 0.9833 - val_loss: 0.0877
Epoch 4/15
140/140 ━━━━━━━━━━━━━━━━━━━━ 32s 231ms/step - accuracy: 0.9621 - loss: 0.1296 - val_accuracy: 1.0000 - val_loss: 0.0577
Epoch 5/15
140/140 ━━━━━━━━━━━━━━━━━━━━ 32s 231ms/step - accuracy: 0.9667 - loss: 0.1099 - val_accuracy: 1.0000 - val_loss: 0.0467
Epoch 6/15
140/140 ━━━━━━━━━━━━━━━━━━━━ 32s 230ms/step - accuracy: 0.9701 - loss: 0.0970 - val_accuracy: 1.0000 - val_loss: 0.0443
Epoch 7/15
140/140 ━━━━━━━━━━━━━━━━━━━━ 32s 229ms/step - accuracy: 0.97

In [18]:
import tensorflow as tf
import numpy as np

# =========================
# LOAD MODEL + CLASSES
# =========================
model = tf.keras.models.load_model("waste_classifier_v2.keras")
class_names = np.load("classes.npy", allow_pickle=True)

IMG_SIZE = (224, 224)

# =========================
# PREDICT FUNCTION
# =========================
def predict_image(img_path):

    img = tf.keras.utils.load_img(img_path, target_size=IMG_SIZE)
    img_array = tf.keras.utils.img_to_array(img)

    # ⚠️ IMPORTANT: NO normalization here
    img_array = np.expand_dims(img_array, axis=0)

    preds = model.predict(img_array)

    class_index = np.argmax(preds)
    confidence = np.max(preds)

    print("Predicted Class:", class_names[class_index])
    print("Confidence:", float(confidence))


# =========================
# TEST
# =========================
predict_image(r"C:\Users\paula\Downloads\paper==.jpg")

1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 739ms/step
Predicted Class: paper
Confidence: 0.9698918461799622
